# Fire-event extents and Sentinel-1/Sentinel-2 data preparation

Prepare fire polygons, equal-area unburned candidate rings, EO acquisition manifests and Earth Engine exports.

Run the configuration cell first. See the repository README for data, dependencies and setup.


In [ ]:
# Shared project configuration (adjust for your environment).
import os
from pathlib import Path

# For Colab: mount Google Drive using the next cell, if needed.
# For local use: set FIRE_PROJECT_DIR to your own data directory.
PROJECT_DIR = Path(os.environ.get(
    "FIRE_PROJECT_DIR", "/content/drive/MyDrive/Fire detection"
)).expanduser()
INPUT_GPKG = PROJECT_DIR / "spain-portugal-topfires-2020-2024.gpkg"
OUTPUT_GPKG = PROJECT_DIR / "fire_collection_areas.gpkg"
EO_DIR = PROJECT_DIR / "eo_data"

print(f"Project directory: {PROJECT_DIR}")


In [ ]:
# Optional for Google Colab only; no existing directories are removed.
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


# Read the geopackage top fire

## Configuration

In [ ]:
from pathlib import Path

# ---------- Paths ----------
PROJECT_DIR = PROJECT_DIR
GPKG_PATH = INPUT_GPKG
FIRE_LAYER = "top_fires_2020_2024"

OUTPUT_DIR = PROJECT_DIR
OUTPUT_GPKG = OUTPUT_DIR / "fire_collection_areas.gpkg"

# ---------- Spatial configuration ----------
PIXEL_SIZE_M = 10

# Surrounding ring area / original polygon area
UNBURNED_AREA_RATIO = 1.0

# No separation between fire and surrounding ring
GAP_M = 0

# Buffer area matching tolerance: 0.1%
AREA_RELATIVE_TOLERANCE = 0.001

# ---------- Sentinel-2 configuration ----------
S2_COLLECTION = "COPERNICUS/S2_SR_HARMONIZED"
S2_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]
S2_INDICES = ["NBR", "NDVI", "NDMI"]

# Initial search windows; used in later collection blocks
PRE_FIRE_SEARCH_DAYS = 20
POST_FIRE_SEARCH_DAYS = 20

# ---------- Sentinel-1 configuration ----------
S1_COLLECTION = "COPERNICUS/S1_GRD"
S1_POLARIZATIONS = ["VV", "VH"]
S1_MODE = "IW"

# ---------- Reproducibility ----------
RANDOM_SEED = 42

## Install packages and mount Drive

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## Read and check the fire polygons

In [ ]:
if not GPKG_PATH.exists():
    raise FileNotFoundError(f"File not found: {GPKG_PATH}")

fires = gpd.read_file(GPKG_PATH, layer=FIRE_LAYER)

required_fields = [
    "area_name", "country",
    "detected_datetime", "extinguished_datetime"
]

missing_fields = [
    field for field in required_fields
    if field not in fires.columns
]

if missing_fields:
    raise ValueError(f"Missing fields: {missing_fields}")

if fires.empty:
    raise ValueError("The fire layer is empty.")

if fires.crs is None:
    raise ValueError("The layer has no coordinate reference system.")

bad_geometry = (
    fires.geometry.isna()
    | fires.geometry.is_empty
    | ~fires.geometry.is_valid
)

if bad_geometry.any():
    raise ValueError(
        "Inspect invalid/missing geometries:\n"
        + fires.loc[bad_geometry, ["area_name"]].to_string(index=False)
    )

if not fires.geometry.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    raise ValueError("All fire geometries must be Polygon or MultiPolygon.")

fires = fires.to_crs("EPSG:4326").reset_index(drop=True)

# Preserve an existing identifier, otherwise create one.
if "fire_id" not in fires.columns:
    fires["fire_id"] = [
        f"FIRE_{i + 1:03d}" for i in range(len(fires))
    ]

if fires["fire_id"].isna().any() or fires["fire_id"].duplicated().any():
    raise ValueError("fire_id must be non-null and unique.")

for field in ["detected_datetime", "extinguished_datetime"]:
    fires[field] = pd.to_datetime(fires[field], errors="raise")

print(f"Loaded {len(fires)} fires")
display(fires[["fire_id", "area_name", "country"]])

## Create equal-area surrounding rings and collection extents

In [ ]:
# ============================================================
# BLOCK 4 — MEMORY-EFFICIENT EQUAL-AREA RINGS
#           AND COLLECTION EXTENTS
# ============================================================

import gc
import numpy as np
import geopandas as gpd

from pyproj import Transformer
from shapely.geometry import box
from shapely.ops import transform as shapely_transform


# ============================================================
# SETTINGS
# ============================================================

# Simplification is used ONLY while searching for buffer distance.
# Final buffer/ring geometry still uses the original polygon.
SEARCH_SIMPLIFY_TOLERANCE_M = max(
    PIXEL_SIZE_M * 0.5,
    1.0
)

# 30 iterations are already far more than sufficient for
# practical spatial precision.
MAX_SEARCH_ITERATIONS = 30

# Final exact refinement iterations using original geometry.
MAX_EXACT_REFINEMENT = 10

# Set False if you do not actually need the full outer buffer layer.
# This saves additional RAM.
KEEP_BUFFER_LAYER = False


# ============================================================
# FAST GEOMETRY TRANSFORMATION
# ============================================================

def transform_geometry(
    geometry,
    source_crs,
    target_crs
):
    """
    Transform one Shapely geometry without creating temporary
    GeoSeries objects.
    """

    transformer = Transformer.from_crs(
        source_crs,
        target_crs,
        always_xy=True
    )

    return shapely_transform(
        transformer.transform,
        geometry
    )


# ============================================================
# FIND BUFFER DISTANCE
# ============================================================

def find_buffer_distance(
    geometry,
    area_ratio,
    tolerance,
    simplify_tolerance
):
    """
    Estimate a buffer distance whose additional area equals
    fire_area * area_ratio.

    A simplified copy is used during the first search to reduce
    RAM and processing requirements. The original geometry is
    then used for final refinement.
    """

    if area_ratio <= 0:
        raise ValueError(
            "area_ratio must be positive."
        )

    # --------------------------------------------------------
    # ORIGINAL TARGET AREA
    # --------------------------------------------------------

    fire_area = geometry.area

    target_ring_area = (
        fire_area *
        area_ratio
    )


    # --------------------------------------------------------
    # SIMPLIFIED GEOMETRY FOR INITIAL SEARCH
    # --------------------------------------------------------

    search_geometry = geometry.simplify(
        simplify_tolerance,
        preserve_topology=True
    )

    search_fire_area = (
        search_geometry.area
    )


    def search_ring_area(distance):

        buffered = search_geometry.buffer(
            distance
        )

        area = (
            buffered.area -
            search_fire_area
        )

        # explicitly release large temporary geometry
        del buffered

        return area


    # --------------------------------------------------------
    # FIND UPPER BOUND
    # --------------------------------------------------------

    lower = 0.0
    upper = 100.0

    while (
        search_ring_area(upper)
        <
        target_ring_area
    ):

        upper *= 2.0

        if upper > 1_000_000:
            raise RuntimeError(
                "Could not determine a reasonable "
                "upper buffer distance."
            )


    # --------------------------------------------------------
    # INITIAL BINARY SEARCH ON SIMPLIFIED GEOMETRY
    # --------------------------------------------------------

    for _ in range(
        MAX_SEARCH_ITERATIONS
    ):

        distance = (
            lower + upper
        ) / 2.0

        current_area = (
            search_ring_area(
                distance
            )
        )

        relative_error = (
            abs(
                current_area -
                target_ring_area
            )
            /
            target_ring_area
        )

        if relative_error <= tolerance:
            break

        if (
            current_area <
            target_ring_area
        ):

            lower = distance

        else:

            upper = distance


    estimated_distance = (
        lower + upper
    ) / 2.0


    # Release simplified geometry before exact processing
    del search_geometry
    gc.collect()


    # ========================================================
    # EXACT REFINEMENT USING ORIGINAL GEOMETRY
    # ========================================================

    # Establish a narrow bracket around the estimate.
    exact_lower = max(
        0.0,
        estimated_distance * 0.75
    )

    exact_upper = (
        estimated_distance * 1.25
    )


    def exact_ring_area(distance):

        buffered = geometry.buffer(
            distance
        )

        area = (
            buffered.area -
            fire_area
        )

        del buffered

        return area


    # Make sure upper bound is actually large enough
    while (
        exact_ring_area(
            exact_upper
        )
        <
        target_ring_area
    ):

        exact_upper *= 1.5


    for _ in range(
        MAX_EXACT_REFINEMENT
    ):

        distance = (
            exact_lower +
            exact_upper
        ) / 2.0

        current_area = (
            exact_ring_area(
                distance
            )
        )

        relative_error = (
            abs(
                current_area -
                target_ring_area
            )
            /
            target_ring_area
        )

        if relative_error <= tolerance:

            return distance


        if (
            current_area <
            target_ring_area
        ):

            exact_lower = distance

        else:

            exact_upper = distance


    return (
        exact_lower +
        exact_upper
    ) / 2.0


# ============================================================
# OUTPUT RECORDS
# ============================================================

ring_records = []
extent_records = []

if KEEP_BUFFER_LAYER:
    buffer_records = []


# ============================================================
# PROCESS FIRES ONE AT A TIME
# ============================================================

total_fires = len(fires)


for n, (_, row) in enumerate(
    fires.iterrows(),
    start=1
):

    print(
        f"Processing {n}/{total_fires}: "
        f"{row['fire_id']}"
    )


    # --------------------------------------------------------
    # DETERMINE LOCAL PROJECTED CRS
    # --------------------------------------------------------

    site = gpd.GeoSeries(
        [row.geometry],
        crs=fires.crs
    )

    local_crs = (
        site.estimate_utm_crs()
    )

    if local_crs is None:

        raise ValueError(
            f"Could not determine UTM CRS "
            f"for {row['fire_id']}"
        )


    # --------------------------------------------------------
    # PROJECT FIRE GEOMETRY
    # --------------------------------------------------------

    fire_geometry = transform_geometry(
        row.geometry,
        fires.crs,
        local_crs
    )

    # Temporary GeoSeries no longer needed
    del site


    # --------------------------------------------------------
    # FIRE AREA
    # --------------------------------------------------------

    fire_area_m2 = (
        fire_geometry.area
    )


    # --------------------------------------------------------
    # FIND EQUAL-AREA BUFFER DISTANCE
    # --------------------------------------------------------

    distance = find_buffer_distance(

        geometry=fire_geometry,

        area_ratio=UNBURNED_AREA_RATIO,

        tolerance=AREA_RELATIVE_TOLERANCE,

        simplify_tolerance=(
            SEARCH_SIMPLIFY_TOLERANCE_M
        )
    )


    # --------------------------------------------------------
    # CREATE FINAL EXACT BUFFER
    # --------------------------------------------------------

    buffered_geometry = (
        fire_geometry.buffer(
            distance
        )
    )


    # --------------------------------------------------------
    # SURROUNDING UNBURNED RING
    #
    # No gap:
    # ring touches the original fire perimeter.
    # --------------------------------------------------------

    ring_geometry = (
        buffered_geometry
        .difference(
            fire_geometry
        )
    )


    ring_area_m2 = (
        ring_geometry.area
    )


    # --------------------------------------------------------
    # COLLECTION EXTENT
    # --------------------------------------------------------

    xmin, ymin, xmax, ymax = (
        buffered_geometry.bounds
    )


    # Snap outward to raster pixel grid
    xmin = (
        np.floor(
            xmin /
            PIXEL_SIZE_M
        )
        *
        PIXEL_SIZE_M
    )

    ymin = (
        np.floor(
            ymin /
            PIXEL_SIZE_M
        )
        *
        PIXEL_SIZE_M
    )

    xmax = (
        np.ceil(
            xmax /
            PIXEL_SIZE_M
        )
        *
        PIXEL_SIZE_M
    )

    ymax = (
        np.ceil(
            ymax /
            PIXEL_SIZE_M
        )
        *
        PIXEL_SIZE_M
    )


    extent_geometry = box(
        xmin,
        ymin,
        xmax,
        ymax
    )


    # --------------------------------------------------------
    # ATTRIBUTES
    # --------------------------------------------------------

    attributes = (
        row.drop(
            labels=[
                fires.geometry.name
            ]
        )
        .to_dict()
    )


    attributes.update({

        "processing_crs":
            local_crs.to_string(),

        "buffer_distance_m":
            float(distance),

        "polygon_area_ha":
            fire_area_m2 / 10000,

        "ring_area_ha":
            ring_area_m2 / 10000,

        "ring_to_fire_ratio":
            ring_area_m2 /
            fire_area_m2,

        "pixel_size_m":
            PIXEL_SIZE_M,

        "grid_xmin":
            float(xmin),

        "grid_ymin":
            float(ymin),

        "grid_xmax":
            float(xmax),

        "grid_ymax":
            float(ymax),

        "grid_width_pixels":
            int(
                round(
                    (xmax - xmin)
                    /
                    PIXEL_SIZE_M
                )
            ),

        "grid_height_pixels":
            int(
                round(
                    (ymax - ymin)
                    /
                    PIXEL_SIZE_M
                )
            ),
    })


    # --------------------------------------------------------
    # TRANSFORM FINAL GEOMETRIES TO WGS84
    # --------------------------------------------------------

    ring_wgs84 = transform_geometry(
        ring_geometry,
        local_crs,
        "EPSG:4326"
    )

    extent_wgs84 = transform_geometry(
        extent_geometry,
        local_crs,
        "EPSG:4326"
    )


    ring_records.append({
        **attributes,
        "geometry": ring_wgs84
    })


    extent_records.append({
        **attributes,
        "geometry": extent_wgs84
    })


    # --------------------------------------------------------
    # OPTIONAL BUFFER LAYER
    # --------------------------------------------------------

    if KEEP_BUFFER_LAYER:

        buffer_wgs84 = transform_geometry(
            buffered_geometry,
            local_crs,
            "EPSG:4326"
        )

        buffer_records.append({
            **attributes,
            "geometry": buffer_wgs84
        })

        del buffer_wgs84


    # --------------------------------------------------------
    # CLEAR LARGE TEMPORARY OBJECTS
    # --------------------------------------------------------

    del fire_geometry
    del buffered_geometry
    del ring_geometry
    del extent_geometry
    del ring_wgs84
    del extent_wgs84

    gc.collect()


# ============================================================
# CREATE OUTPUT GEODATAFRAMES
# ============================================================

unburned_rings = gpd.GeoDataFrame(
    ring_records,
    geometry="geometry",
    crs="EPSG:4326"
)


collection_extents = gpd.GeoDataFrame(
    extent_records,
    geometry="geometry",
    crs="EPSG:4326"
)


if KEEP_BUFFER_LAYER:

    buffers = gpd.GeoDataFrame(
        buffer_records,
        geometry="geometry",
        crs="EPSG:4326"
    )


# ============================================================
# CLEAN RECORD LISTS FROM MEMORY
# ============================================================

del ring_records
del extent_records

if KEEP_BUFFER_LAYER:
    del buffer_records

gc.collect()


# ============================================================
# CHECK RESULTS
# ============================================================

check_columns = [
    "fire_id",
    "area_name",
    "processing_crs",
    "polygon_area_ha",
    "ring_area_ha",
    "ring_to_fire_ratio",
    "buffer_distance_m",
    "grid_width_pixels",
    "grid_height_pixels",
]


print("\nProcessing completed.")
print(
    f"Unburned rings: "
    f"{len(unburned_rings)}"
)

print(
    f"Collection extents: "
    f"{len(collection_extents)}"
)


# Convert to ordinary DataFrame before displaying
# so geometry is not accidentally rendered.
result_table = (
    collection_extents[
        check_columns
    ]
    .copy()
    .round(3)
)

print(
    result_table.to_string(
        index=False
    )
)

## Inspect the geometries

In [ ]:
ncols = 3
nrows = int(np.ceil(len(fires) / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(15, 4.5 * nrows),
    squeeze=False
)

for i, ax in enumerate(axes.flat):
    if i >= len(fires):
        ax.axis("off")
        continue

    local_crs = collection_extents.iloc[i]["processing_crs"]

    # Plot in the projected CRS to preserve shape and distance.
    extent = collection_extents.iloc[[i]].to_crs(local_crs)
    ring = unburned_rings.iloc[[i]].to_crs(local_crs)
    fire = fires.iloc[[i]].to_crs(local_crs)

    ring.plot(ax=ax, color="#b8d8ba", edgecolor="none")
    fire.plot(ax=ax, color="#e87561", edgecolor="#a53325", linewidth=0.5)
    extent.boundary.plot(
        ax=ax, color="#264653", linewidth=1, linestyle="--"
    )

    ax.set_title(f"{fires.iloc[i]['area_name']} — {fires.iloc[i]['country']}")
    ax.set_aspect("equal")
    ax.set_axis_off()

fig.suptitle(
    "Red: fire polygon | Green: equal-area surrounding ring | "
    "Dashed: collection extent",
    fontsize=12
)
plt.tight_layout()
plt.show()

Block 6 — Save the preparation results

In [ ]:
if OUTPUT_GPKG.exists():
    OUTPUT_GPKG.unlink()

layers_to_save = {
    "fire_polygons": fires,
    # "buffered_areas": buffers,  # Commented out as KEEP_BUFFER_LAYER is False
    "candidate_unburned_rings": unburned_rings,
    "collection_extents": collection_extents,
}

for layer_name, frame in layers_to_save.items():
    frame.to_file(
        OUTPUT_GPKG,
        layer=layer_name,
        driver="GPKG",
        engine="pyogrio",
    )

summary_path = OUTPUT_DIR / "collection_area_summary.csv"

collection_extents.drop(
    columns=collection_extents.geometry.name
).to_csv(summary_path, index=False)

print(f"Saved GeoPackage: {OUTPUT_GPKG}")
print(f"Saved summary: {summary_path}")

# Load the saved burne/unburned geopackage

In [ ]:
import geopandas as gpd
from pathlib import Path

OUTPUT_GPKG = PROJECT_DIR / 'fire_collection_areas.gpkg'

# Load the layers from the GeoPackage
fire_polygons_loaded = gpd.read_file(OUTPUT_GPKG, layer="fire_polygons")
unburned_rings_loaded = gpd.read_file(OUTPUT_GPKG, layer="candidate_unburned_rings")
collection_extents_loaded = gpd.read_file(OUTPUT_GPKG, layer="collection_extents")

print("Loaded GeoPackage layers:")
print(f"  - fire_polygons: {len(fire_polygons_loaded)} features")
print(f"  - candidate_unburned_rings: {len(unburned_rings_loaded)} features")
print(f"  - collection_extents: {len(collection_extents_loaded)} features")

# EO data preparation

## Install packages

## Configuration

In [ ]:
from pathlib import Path

CFG = {
    # Replace with your Earth Engine-enabled Google Cloud project ID.
    "ee_project": os.environ.get("EE_PROJECT_ID", "YOUR_GOOGLE_CLOUD_PROJECT_ID"),

    # Input polygons: keep this path unchanged.
    "gpkg_path": str(OUTPUT_GPKG),
    "extent_layer": "collection_extents",

    # Earth Engine uses the folder NAME, not its full path.
    "export_folder": "eo_data",

    "pixel_size": 20,

    "s2_bands": [
        "B2", "B3", "B4",
        "B5", "B6", "B7", "B8", "B8A",
        "B11", "B12",
    ],

    # Search windows before detection and after extinction.
    "s2_pre_days": 45,
    "s2_post_days": 45,
    "s1_pre_days": 30,
    "s1_post_days": 30,

    # Minimum usable fraction of the collection extent.
    "s2_min_valid_fraction": 0.99,
    "s1_min_valid_fraction": 0.99,

    # Minimum distinct S1 acquisition dates per period.
    "s1_min_dates": 2,

    "nodata": -9999,
}

# Process one fire first; change to None for all fires.
SITE_LIMIT = None

# Output directory. Block 3 creates it AFTER mounting Drive.
LOCAL_OUTPUT_DIR = PROJECT_DIR / 'eo_data'

MANIFEST_PATH = (
    LOCAL_OUTPUT_DIR / "satellite_selection_manifest.json"
)

TASK_LOG_PATH = (
    LOCAL_OUTPUT_DIR / "satellite_export_tasks.json"
)

## Initialize Earth Engine

In [ ]:
import ee

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Interactive authentication does not expose credentials in the notebook.
ee.Authenticate()
ee.Initialize(project=CFG["ee_project"])
print("Earth Engine initialized.")


## Read the collection extents and check dates

In [ ]:
import geopandas as gpd
import pandas as pd
import pyogrio

print(
    pd.DataFrame(
        pyogrio.list_layers(CFG["gpkg_path"]),
        columns=["layer", "geometry_type"],
    )
)

sites = gpd.read_file(
    CFG["gpkg_path"],
    layer=CFG["extent_layer"],
)

required = [
    "area_name",
    "country",
    "detected_datetime",
    "extinguished_datetime",
]

missing = [c for c in required if c not in sites.columns]
if missing:
    raise ValueError(f"Missing required attributes: {missing}")

if sites.crs is None:
    raise ValueError("The collection extent layer has no CRS.")

for col in ["detected_datetime", "extinguished_datetime"]:
    sites[col] = pd.to_datetime(
        sites[col], errors="coerce", utc=True
    )

bad_dates = (
    sites["detected_datetime"].isna()
    | sites["extinguished_datetime"].isna()
    | (
        sites["extinguished_datetime"]
        < sites["detected_datetime"]
    )
)

if bad_dates.any():
    raise ValueError(
        "Invalid fire dates:\n"
        + sites.loc[bad_dates, required].to_string(index=False)
    )

if sites.geometry.isna().any() or sites.geometry.is_empty.any():
    raise ValueError("Missing or empty collection extents.")

if not sites.geometry.is_valid.all():
    raise ValueError("Invalid collection geometries.")

# Stable identifiers for this file's row order.
sites = sites.reset_index(drop=True)
sites["site_key"] = [
    f"fire_{i + 1:02d}" for i in range(len(sites))
]

display(sites[["site_key"] + required])

## Build the shared grid and date windows

In [ ]:
import math

def make_site_context(row):
    geometry = gpd.GeoSeries(
        [row.geometry], crs=sites.crs
    )

    utm_crs = geometry.estimate_utm_crs()
    xmin, ymin, xmax, ymax = geometry.to_crs(
        utm_crs
    ).total_bounds

    pixel = CFG["pixel_size"]

    # Snap bounds to the same 20 m grid.
    xmin = math.floor(xmin / pixel) * pixel
    ymin = math.floor(ymin / pixel) * pixel
    xmax = math.ceil(xmax / pixel) * pixel
    ymax = math.ceil(ymax / pixel) * pixel

    crs = utm_crs.to_string()

    region = ee.Geometry.Rectangle(
        [xmin, ymin, xmax, ymax],
        proj=crs,
        geodesic=False,
    )

    detection = row["detected_datetime"].floor("D")
    post_start = (
        row["extinguished_datetime"].floor("D")
        + pd.Timedelta(days=1)
    )

    def before(days):
        return [
            (detection - pd.Timedelta(days=days)).strftime("%Y-%m-%d"),
            detection.strftime("%Y-%m-%d"),
        ]

    def after(days):
        return [
            post_start.strftime("%Y-%m-%d"),
            (post_start + pd.Timedelta(days=days)).strftime("%Y-%m-%d"),
        ]

    return {
        "site_key": row["site_key"],
        "area_name": str(row["area_name"]),
        "country": str(row["country"]),
        "region": region,
        "crs": crs,
        "transform": [pixel, 0, xmin, 0, -pixel, ymax],
        "bounds": [xmin, ymin, xmax, ymax],
        "width": int(round((xmax - xmin) / pixel)),
        "height": int(round((ymax - ymin) / pixel)),
        "windows": {
            "S2_pre": before(CFG["s2_pre_days"]),
            "S2_post": after(CFG["s2_post_days"]),
            "S1_pre": before(CFG["s1_pre_days"]),
            "S1_post": after(CFG["s1_post_days"]),
        },
    }


def valid_fraction(mask, context):
    # Count missing coverage as invalid, rather than excluding it.
    mask = mask.rename("valid").unmask(
        0, sameFootprint=False
    )

    result = mask.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=context["region"],
        crs=context["crs"],
        crsTransform=context["transform"],
        maxPixels=1e9,
        tileScale=4,
    ).get("valid").getInfo()

    return float(result) if result is not None else 0.0

## Select Sentinel-2 pre/post acquisitions

In [ ]:
def prepare_s2(image):
    scl = image.select("SCL")

    # Exclude defective pixels, cloud shadows, clouds,
    # cirrus and snow. Retain dark pixels because burns
    # can be dark.
    usable = scl.remap(
        [2, 4, 5, 6, 7],
        [1, 1, 1, 1, 1],
        0,
    ).eq(1)

    spectral = image.select(CFG["s2_bands"])

    all_bands_present = spectral.mask().reduce(
        ee.Reducer.min()
    )

    usable = usable.And(all_bands_present)

    reflectance = (
        spectral
        .multiply(0.0001)
        .updateMask(usable)
        .resample("bilinear")
        .toFloat()
    )

    return (
        reflectance
        .addBands(scl.rename("SCL").toFloat())
        .copyProperties(image, ["system:time_start"])
    )


def select_s2(context, period):
    start, end = context["windows"][period]

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(context["region"])
        .filterDate(start, end)
        .map(
            lambda image: image.set(
                "acquisition_day",
                image.date().format("YYYY-MM-dd"),
            )
        )
    )

    days = sorted(
        set(collection.aggregate_array("acquisition_day").getInfo()),
        reverse=(period == "S2_pre"),
    )

    checked = []

    for day in days:
        daily = collection.filter(
            ee.Filter.eq("acquisition_day", day)
        )

        image = (
            daily.sort("system:index")
            .map(prepare_s2)
            .mosaic()
        )

        valid = image.select(
            CFG["s2_bands"]
        ).mask().reduce(ee.Reducer.min())

        fraction = valid_fraction(valid, context)

        checked.append({
            "date": day,
            "valid_fraction": fraction,
        })

        if fraction >= CFG["s2_min_valid_fraction"]:
            output = (
                image
                .addBands(
                    valid.unmask(0, sameFootprint=False)
                    .rename("valid")
                    .toFloat()
                )
                .clip(context["region"])
            )

            metadata = {
                "date": day,
                "valid_fraction": fraction,
                "source_ids": daily.aggregate_array(
                    "system:index"
                ).getInfo(),
                "checked_candidates": checked,
            }

            return output, metadata

    raise RuntimeError(
        f"{period}: no acquisition meets the valid-area criterion "
        f"in {start} to {end}. Candidates: {checked}"
    )

## Select a common Sentinel-1 orbit and make composites

In [ ]:
def get_s1_collection(context, period):
    start, end = context["windows"][period]

    return (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(context["region"])
        .filterDate(start, end)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.eq("resolution_meters", 10))
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation", "VV"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation", "VH"
            )
        )
    )


def select_s1_pair(context):
    pre = get_s1_collection(context, "S1_pre")
    post = get_s1_collection(context, "S1_post")

    def stamp(image):
        return image.set({
            "track_key": ee.String(
                image.get("orbitProperties_pass")
            ).cat("_").cat(
                ee.Number(
                    image.get("relativeOrbitNumber_start")
                ).format("%d")
            ),
            "acquisition_day": image.date().format("YYYY-MM-dd"),
        })

    pre = pre.map(stamp)
    post = post.map(stamp)

    common_tracks = sorted(
        set(pre.aggregate_array("track_key").getInfo())
        & set(post.aggregate_array("track_key").getInfo())
    )

    candidates = []

    for track in common_tracks:
        collections = {
            "S1_pre": pre.filter(ee.Filter.eq("track_key", track)),
            "S1_post": post.filter(ee.Filter.eq("track_key", track)),
        }

        images = {}
        metadata = {}

        for period, collection in collections.items():
            dates = sorted(set(
                collection.aggregate_array("acquisition_day").getInfo()
            ))

            if len(dates) < CFG["s1_min_dates"]:
                break

            def prepare(image):
                bands = image.select(["VV", "VH"])
                joint_mask = bands.mask().reduce(
                    ee.Reducer.min()
                )
                return bands.updateMask(joint_mask).resample(
                    "bilinear"
                )

            prepared = collection.map(prepare)
            median = prepared.median().toFloat()

            valid = median.mask().reduce(ee.Reducer.min())
            fraction = valid_fraction(valid, context)

            images[period] = (
                median
                .addBands(
                    prepared.select("VV").count()
                    .unmask(0, sameFootprint=False)
                    .rename("n_obs").toFloat()
                )
                .addBands(
                    valid.unmask(0, sameFootprint=False)
                    .rename("valid").toFloat()
                )
                .clip(context["region"])
            )

            metadata[period] = {
                "track": track,
                "dates": dates,
                "number_of_dates": len(dates),
                "valid_fraction": fraction,
                "source_ids": collection.aggregate_array(
                    "system:index"
                ).getInfo(),
            }

        if len(images) == 2:
            coverage = min(
                metadata[p]["valid_fraction"] for p in images
            )
            minimum_dates = min(
                metadata[p]["number_of_dates"] for p in images
            )

            if coverage >= CFG["s1_min_valid_fraction"]:
                candidates.append(
                    (coverage, minimum_dates, track, images, metadata)
                )

    if not candidates:
        raise RuntimeError(
            "No common S1 track satisfies coverage and "
            "minimum-acquisition requirements in both periods."
        )

    # Prefer complete coverage, then more acquisitions.
    candidates.sort(
        key=lambda x: (x[0], x[1], x[2]),
        reverse=True,
    )

    _, _, _, images, metadata = candidates[0]
    return images, metadata

## Run selection and save the acquisition manifest

In [ ]:
import json

selected = {}
manifest = []
failures = []

working_sites = (
    sites if SITE_LIMIT is None else sites.head(SITE_LIMIT)
)

for _, row in working_sites.iterrows():
    key = row["site_key"]
    print(f"\nSelecting {key}: {row['area_name']}")

    try:
        context = make_site_context(row)

        s2_pre, s2_pre_meta = select_s2(context, "S2_pre")
        s2_post, s2_post_meta = select_s2(context, "S2_post")
        s1_images, s1_meta = select_s1_pair(context)

        images = {
            "S2_pre": s2_pre,
            "S2_post": s2_post,
            **s1_images,
        }

        selected[key] = {
            "context": context,
            "images": images,
        }

        record = {
            k: v for k, v in context.items() if k != "region"
        }
        record["acquisitions"] = {
            "S2_pre": s2_pre_meta,
            "S2_post": s2_post_meta,
            **s1_meta,
        }
        manifest.append(record)

        print(
            "S2:", s2_pre_meta["date"], "→", s2_post_meta["date"],
            "| S1 track:", s1_meta["S1_pre"]["track"],
        )

    except Exception as error:
        failures.append({
            "site_key": key,
            "area_name": str(row["area_name"]),
            "error": str(error),
        })
        print("REQUIRES REVIEW:", error)

    # Save progress after every site.
    MANIFEST_PATH.write_text(
        json.dumps(
            {
                "configuration": CFG,
                "selected_sites": manifest,
                "failures": failures,
            },
            indent=2,
        ),
        encoding="utf-8",
    )

print(f"\nSelected: {len(selected)}")
print(f"Requires review: {len(failures)}")
print("Manifest:", MANIFEST_PATH)

## Export clipped GeoTIFFs to Drive

In [ ]:
tasks = []
task_records = []

for key, result in selected.items():
    context = result["context"]

    for product, image in result["images"].items():
        name = f"{key}_{product}_20m"

        task = ee.batch.Export.image.toDrive(
            image=image.toFloat().unmask(
                CFG["nodata"], sameFootprint=False
            ),
            description=name,
            folder=CFG["export_folder"],
            fileNamePrefix=name,
            region=context["region"],
            crs=context["crs"],
            crsTransform=context["transform"],
            maxPixels=1e10,
            fileFormat="GeoTIFF",
            formatOptions={
                "cloudOptimized": True,
                "noData": CFG["nodata"],
            },
        )

        task.start()
        tasks.append(task)

        task_records.append({
            "task_id": task.id,
            "site_key": key,
            "product": product,
            "filename_prefix": name,
        })

        TASK_LOG_PATH.write_text(
            json.dumps(task_records, indent=2),
            encoding="utf-8",
        )

        print("Submitted:", name, task.id)

## Check export progress

In [ ]:
records = json.loads(TASK_LOG_PATH.read_text())

statuses = ee.data.getTaskStatus(
    [record["task_id"] for record in records]
)

status_by_id = {status["id"]: status for status in statuses}

display(pd.DataFrame([
    {
        "site": record["site_key"],
        "product": record["product"],
        "state": status_by_id.get(
            record["task_id"], {}
        ).get("state", "UNKNOWN"),
        "error": status_by_id.get(
            record["task_id"], {}
        ).get("error_message", ""),
    }
    for record in records
]))

# Load saved S1/S2 rasters

## Install and import

In [ ]:

from pathlib import Path
import re
import numpy as np
import pandas as pd
import rasterio
from rasterio.merge import merge

## Find the exported rasters

In [ ]:
EO_DIR = PROJECT_DIR / 'eo_data'

if not EO_DIR.exists():
    raise FileNotFoundError(f"Folder not found: {EO_DIR}")

PRODUCTS = ["S2_pre", "S2_post", "S1_pre", "S1_post"]

BAND_NAMES = {
    "S2_pre": [
        "B2", "B3", "B4", "B5", "B6",
        "B7", "B8", "B8A", "B11", "B12",
        "SCL", "valid",
    ],
    "S2_post": [
        "B2", "B3", "B4", "B5", "B6",
        "B7", "B8", "B8A", "B11", "B12",
        "SCL", "valid",
    ],
    "S1_pre": ["VV", "VH", "n_obs", "valid"],
    "S1_post": ["VV", "VH", "n_obs", "valid"],
}

pattern = re.compile(
    r"^(fire_\d+)_(S2_pre|S2_post|S1_pre|S1_post)"
    r"_20m(?:-\d+-\d+)?\.tiff?$",
    re.IGNORECASE,
)

# catalog[fire_id][product] = list of raster paths
catalog = {}
unrecognized = []

for path in sorted(EO_DIR.iterdir()):
    if path.suffix.lower() not in {".tif", ".tiff"}:
        continue

    match = pattern.match(path.name)

    if not match:
        unrecognized.append(path.name)
        continue

    fire_id = match.group(1).lower()
    product = next(
        p for p in PRODUCTS
        if p.lower() == match.group(2).lower()
    )

    catalog.setdefault(fire_id, {}).setdefault(
        product, []
    ).append(path)

if not catalog:
    raise FileNotFoundError("No matching exported rasters found.")

summary = pd.DataFrame([
    {
        "fire_id": fire_id,
        **{
            product: len(files.get(product, []))
            for product in PRODUCTS
        },
        "complete": all(
            product in files for product in PRODUCTS
        ),
    }
    for fire_id, files in sorted(catalog.items())
])

display(summary)

if unrecognized:
    print("Unrecognized TIFF filenames:", unrecognized)

## Inspect metadata without loading pixel arrays

In [ ]:
metadata_rows = []

for fire_id, products in sorted(catalog.items()):
    for product, paths in products.items():
        for path in paths:
            with rasterio.open(path) as src:
                metadata_rows.append({
                    "fire_id": fire_id,
                    "product": product,
                    "filename": path.name,
                    "bands": src.count,
                    "width": src.width,
                    "height": src.height,
                    "crs": str(src.crs),
                    "pixel_size": src.res,
                    "nodata": src.nodata,
                    "size_MB": round(
                        path.stat().st_size / 1024**2, 2
                    ),
                })

raster_metadata = pd.DataFrame(metadata_rows)
display(raster_metadata)

## Define the raster loader

In [ ]:
from contextlib import ExitStack

def load_product(fire_id, product):
    paths = catalog.get(fire_id, {}).get(product)

    if not paths:
        raise FileNotFoundError(
            f"Missing {product} for {fire_id}"
        )

    with ExitStack() as stack:
        sources = [
            stack.enter_context(rasterio.open(path))
            for path in paths
        ]

        reference = sources[0]
        expected_bands = BAND_NAMES[product]

        for src in sources:
            if src.count != len(expected_bands):
                raise ValueError(
                    f"{src.name}: expected "
                    f"{len(expected_bands)} bands, "
                    f"found {src.count}"
                )

            if (
                src.crs != reference.crs
                or src.res != reference.res
                or src.nodata != reference.nodata
            ):
                raise ValueError(
                    f"Inconsistent raster pieces for "
                    f"{fire_id}, {product}"
                )

        if len(sources) == 1:
            data = reference.read(
                masked=True
            ).astype("float32").filled(np.nan)

            transform = reference.transform

        else:
            nodata = reference.nodata
            if nodata is None:
                nodata = -9999.0

            data, transform = merge(
                sources,
                nodata=nodata,
                dtype="float32",
            )

            data[data == nodata] = np.nan

        return {
            "data": data,
            "band_names": expected_bands,
            "crs": reference.crs,
            "transform": transform,
            "height": data.shape[1],
            "width": data.shape[2],
            "paths": [str(path) for path in paths],
        }


def load_fire(fire_id):
    rasters = {
        product: load_product(fire_id, product)
        for product in PRODUCTS
    }

    reference = rasters["S2_pre"]

    for product, raster in rasters.items():
        if (
            raster["crs"] != reference["crs"]
            or raster["transform"] != reference["transform"]
            or raster["data"].shape[1:]
            != reference["data"].shape[1:]
        ):
            raise ValueError(
                f"{fire_id}: {product} does not match "
                "the S2_pre grid."
            )

    print(f"{fire_id}: all four grids match.")

    return rasters

## PRE/POST IMAGE AND REFERENCE-MASK ILLUSTRATION

In [ ]:
# ============================================================
# BLOCK 16 — COMPACT PRE/POST IMAGE COMPARISON
# Requires Blocks 11–14: EO_DIR, catalog and load_product.
# ============================================================

import json
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path
from rasterio.features import rasterize
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

FIGURE_DIR = PROJECT_DIR / 'print_layouts'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_GPKG = (
    str(PROJECT_DIR) + "/"
    "fire_collection_areas.gpkg"
)
REFERENCE_LAYER = "fire_polygons"

MANIFEST_FILE = EO_DIR / "satellite_selection_manifest.json"

FIRE_IDS = None                  # None = all available S2 pairs
FIRES_PER_FIGURE = 6
MAX_DISPLAY_PIXELS = 800
DPI = 300

TRUE_COLOR = ["B4", "B3", "B2"]
FALSE_COLOR = ["B12", "B8", "B4"]

# Fixed display ranges for all fires and both dates.
TRUE_RANGE = (0.0, 0.30)
FALSE_RANGE = (0.0, 0.50)

GAMMA = 1.2
SATURATION = 0.80

# Font sizes.
COLUMN_FONT_SIZE = 14
ROW_FONT_SIZE = 12
LEGEND_FONT_SIZE = 11
DATE_FONT_SIZE = 9

# Layout dimensions in inches.
PANEL_WIDTH = 2.0
COLUMN_GAP = 0.05
ROW_GAP = 0.18
LEFT_MARGIN = 1.0
RIGHT_MARGIN = 0.15
TOP_MARGIN = 0.85
BOTTOM_MARGIN = 0.85

COLUMN_TITLES = [
    "Pre-fire\nTrue color",
    "Pre-fire\nFalse color",
    "Post-fire\nTrue color",
    "Post-fire\nFalse color",
    "Reference\nperimeter mask",
]

INVALID_COLOR = "#cccccc"


# ------------------------------------------------------------
# 2. READ MANIFEST AND REFERENCE POLYGONS
# ------------------------------------------------------------

manifest = json.loads(MANIFEST_FILE.read_text())

site_info = {
    record["site_key"]: record
    for record in manifest["selected_sites"]
}

reference_polygons = gpd.read_file(
    REFERENCE_GPKG,
    layer=REFERENCE_LAYER,
)

if reference_polygons.crs is None:
    raise ValueError("Reference polygons have no CRS.")

if FIRE_IDS is None:
    FIRE_IDS = [
        fire_id
        for fire_id in sorted(catalog)
        if "S2_pre" in catalog[fire_id]
        and "S2_post" in catalog[fire_id]
    ]

if not FIRE_IDS:
    raise ValueError("No complete Sentinel-2 pairs were found.")


def get_reference_polygon(fire_id, target_crs):
    info = site_info[fire_id]

    matches = reference_polygons[
        (
            reference_polygons["area_name"].astype(str)
            == info["area_name"]
        )
        & (
            reference_polygons["country"].astype(str)
            == info["country"]
        )
    ]

    if len(matches) != 1:
        raise ValueError(
            f"{fire_id}: expected one reference polygon, "
            f"found {len(matches)}."
        )

    geometry = matches.to_crs(target_crs).geometry.iloc[0]

    if (
        geometry is None
        or geometry.is_empty
        or not geometry.is_valid
    ):
        raise ValueError(f"{fire_id}: invalid reference geometry.")

    return geometry


# ------------------------------------------------------------
# 3. DISPLAY FUNCTIONS
# ------------------------------------------------------------

def get_display_valid(raster, stride):
    valid_index = raster["band_names"].index("valid")

    valid = (
        raster["data"][valid_index, ::stride, ::stride] > 0.5
    )

    spectral = raster["data"][:10, ::stride, ::stride]

    return valid & np.isfinite(spectral).all(axis=0)


def make_rgb(raster, bands, stride, display_range):
    indexes = [
        raster["band_names"].index(band)
        for band in bands
    ]

    rgb = raster["data"][
        indexes, ::stride, ::stride
    ].transpose(1, 2, 0).copy()

    valid = get_display_valid(raster, stride)

    low, high = display_range

    if high <= low:
        raise ValueError("Display range must have high > low.")

    rgb = np.clip((rgb - low) / (high - low), 0, 1)
    rgb = np.nan_to_num(rgb, nan=0.0)
    rgb = rgb ** (1.0 / GAMMA)

    gray = (
        0.2126 * rgb[..., 0]
        + 0.7152 * rgb[..., 1]
        + 0.0722 * rgb[..., 2]
    )[..., None]

    rgb = gray + SATURATION * (rgb - gray)
    rgb = np.clip(rgb, 0, 1).astype("float32")

    rgb[~valid] = [0.8, 0.8, 0.8]

    return rgb


mask_cmap = ListedColormap(["black", "#b30000"])
mask_cmap.set_bad(INVALID_COLOR)


def prepare_fire_panels(fire_id):
    pre = load_product(fire_id, "S2_pre")
    post = load_product(fire_id, "S2_post")

    if (
        pre["crs"] != post["crs"]
        or pre["transform"] != post["transform"]
        or pre["data"].shape[1:] != post["data"].shape[1:]
    ):
        raise ValueError(
            f"{fire_id}: Sentinel-2 pre/post grids do not match."
        )

    stride = max(
        1,
        int(np.ceil(
            max(pre["height"], pre["width"])
            / MAX_DISPLAY_PIXELS
        )),
    )

    panels = [
        make_rgb(pre, TRUE_COLOR, stride, TRUE_RANGE),
        make_rgb(pre, FALSE_COLOR, stride, FALSE_RANGE),
        make_rgb(post, TRUE_COLOR, stride, TRUE_RANGE),
        make_rgb(post, FALSE_COLOR, stride, FALSE_RANGE),
    ]

    geometry = get_reference_polygon(fire_id, pre["crs"])

    reference_mask = rasterize(
        [(geometry, 1)],
        out_shape=(pre["height"], pre["width"]),
        transform=pre["transform"],
        fill=0,
        all_touched=False,
        dtype="uint8",
    )[::stride, ::stride]

    common_valid = (
        get_display_valid(pre, stride)
        & get_display_valid(post, stride)
    )

    panels.append(
        np.ma.array(
            reference_mask,
            mask=~common_valid,
        )
    )

    info = site_info[fire_id]
    display_height, display_width = panels[0].shape[:2]

    return {
        "fire_id": fire_id,
        "name": info["area_name"],
        "country": info["country"],
        "pre_date": info["acquisitions"]["S2_pre"]["date"],
        "post_date": info["acquisitions"]["S2_post"]["date"],
        "aspect_ratio": display_height / display_width,
        "panels": panels,
    }


# ------------------------------------------------------------
# 4. CREATE FIGURES
# ------------------------------------------------------------

for page_start in range(0, len(FIRE_IDS), FIRES_PER_FIGURE):
    page_ids = FIRE_IDS[
        page_start:page_start + FIRES_PER_FIGURE
    ]

    page_number = page_start // FIRES_PER_FIGURE + 1

    rows = []

    for fire_id in page_ids:
        print("Preparing:", fire_id)
        rows.append(prepare_fire_panels(fire_id))

    row_heights = [
        PANEL_WIDTH * row["aspect_ratio"]
        for row in rows
    ]

    figure_width = (
        LEFT_MARGIN
        + 5 * PANEL_WIDTH
        + 4 * COLUMN_GAP
        + RIGHT_MARGIN
    )

    figure_height = (
        TOP_MARGIN
        + sum(row_heights)
        + (len(rows) - 1) * ROW_GAP
        + BOTTOM_MARGIN
    )

    fig = plt.figure(
        figsize=(figure_width, figure_height),
        facecolor="white",
    )

    top_position = figure_height - TOP_MARGIN

    for row_number, (row, row_height) in enumerate(
        zip(rows, row_heights)
    ):
        bottom_position = top_position - row_height

        # Vertical label, centered beside the row.
        row_label = fig.text(
            (LEFT_MARGIN - 0.30) / figure_width,
            (bottom_position + row_height / 2) / figure_height,
            f"{row['name']}\n{row['country']}",
            rotation=90,
            rotation_mode="anchor",
            ha="center",
            va="center",
            fontsize=ROW_FONT_SIZE,
            fontweight="bold",
        )

        for column, panel in enumerate(row["panels"]):
            left_position = (
                LEFT_MARGIN
                + column * (PANEL_WIDTH + COLUMN_GAP)
            )

            ax = fig.add_axes([
                left_position / figure_width,
                bottom_position / figure_height,
                PANEL_WIDTH / figure_width,
                row_height / figure_height,
            ])

            if column < 4:
                ax.imshow(
                    panel,
                    interpolation="nearest",
                    aspect="equal",
                )

                date = (
                    row["pre_date"]
                    if column < 2
                    else row["post_date"]
                )

                ax.text(
                    0.02, 0.02, date,
                    transform=ax.transAxes,
                    fontsize=DATE_FONT_SIZE,
                    color="white",
                    ha="left",
                    va="bottom",
                    bbox={
                        "facecolor": "black",
                        "alpha": 0.65,
                        "edgecolor": "none",
                        "pad": 2,
                    },
                )

            else:
                ax.imshow(
                    panel,
                    cmap=mask_cmap,
                    vmin=0,
                    vmax=1,
                    interpolation="nearest",
                    aspect="equal",
                )

            if row_number == 0:
                ax.set_title(
                    COLUMN_TITLES[column],
                    fontsize=COLUMN_FONT_SIZE,
                    fontweight="bold",
                    pad=12,
                )

            ax.set_axis_off()

        # Fit long vertical labels within short rows.
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()

        available_height_px = (
            row_height * fig.dpi * 0.95
        )

        while (
            row_label.get_window_extent(renderer).height
            > available_height_px
            and row_label.get_fontsize() > 8
        ):
            row_label.set_fontsize(
                row_label.get_fontsize() - 0.5
            )
            fig.canvas.draw()

        top_position = bottom_position - ROW_GAP

    fig.legend(
        handles=[
            Patch(
                facecolor="#b30000",
                label="Inside fire perimeter",
            ),
            Patch(
                facecolor="black",
                label="Outside fire perimeter",
            ),
            Patch(
                facecolor=INVALID_COLOR,
                label="Invalid optical coverage",
            ),
        ],
        loc="lower center",
        bbox_to_anchor=(0.54, 0.015),
        ncol=3,
        frameon=False,
        prop={
            "size": LEGEND_FONT_SIZE,
            "weight": "bold",
        },
        handlelength=1.8,
        handleheight=1.0,
        columnspacing=1.5,
    )

    prefix = (
        FIGURE_DIR
        / f"fire_pre_post_panels_vertical_{page_number:02d}"
    )

    for extension in [".png", ".pdf"]:
        output_path = prefix.with_suffix(extension)

        fig.savefig(
            output_path,
            dpi=DPI,
            bbox_inches="tight",
            pad_inches=0.08,
            facecolor="white",
        )

        print("Saved:", output_path)

    plt.show()
    plt.close(fig)

    del rows